Instalação de pacotes, seguido de importação de bibliotecas e funções necessárias.

In [1]:
# Instalar ou atualizar bibliotecas necessárias: datasets do Hugging Face, transformers, torch e google.
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

# Importar biblioteca pandas, função genai da biblioteca google e função userdata da biblioteca google.colab.
import pandas as pd
import torch
import huggingface_hub
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.1 MB/s eta 0:00:00


Processar o arquivo de base com as perguntas.

In [2]:
# Carregar as questões a serem usadas, cuja lógica está em outro arquivo.
# Realizar download do arquivo direto do GitHub.
!wget https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb -O questions.ipynb

# Executar o notebook de base na íntegra.
%run questions.ipynb

--2026-04-04 00:48:54--  https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 36004 (35K) [text/plain]
Saving to: ‘questions.ipynb’

questions.ipynb     100%[===================>]  35.16K  --.-KB/s    in 0s      

2026-04-04 00:48:54 (122 MB/s) - ‘questions.ipynb’ saved [36004/36004]



TimeoutException: Requesting secret groq_api_key timed out. Secrets can only be fetched when running from the Colab UI.

TimeoutException: Requesting secret groq_api_key timed out. Secrets can only be fetched when running from the Colab UI.

In [5]:
from google.colab import userdata
from huggingface_hub import login

# Resgata o segredo cadastrado com o nome 'hugging_colab'
try:
    hf_token = userdata.get('hugging_colab')

    # Realiza o login no Hugging Face
    login(token=hf_token)
    print("Conectado com sucesso ao Hugging Face!")
except Exception as e:
    print(f"Erro ao acessar o secret: {e}")
    print("Verifique se o nome do segredo está correto e se o acesso ao notebook foi permitido.")
model_id = "rhaymison/Llama-3-portuguese-Tom-cat-8b-instruct"



# Configuração para carregar em 4 bits (ocupa ~5.5GB de VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

prompt = """Analise a seguinte questão da OAB e responda apenas a alternativa correta:

Questão: A pessoa jurídica 'Alfa' possui um débito tributário de ICMS. O Estado de origem decide perdoar a dívida através de uma lei específica.
Esse instituto de exclusão do crédito tributário é chamado de:
A) Isenção.
B) Anistia.
C) Remissão.
D) Moratória.

Resposta:"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Conectado com sucesso ao Hugging Face!


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Analise a seguinte questão da OAB e responda apenas a alternativa correta:

Questão: A pessoa jurídica 'Alfa' possui um débito tributário de ICMS. O Estado de origem decide perdoar a dívida através de uma lei específica.
Esse instituto de exclusão do crédito tributário é chamado de:
A) Isenção.
B) Anistia.
C) Remissão.
D) Moratória.

Resposta: C) Remissão.

Justificativa: A remissão é a extinção da dívida tributária, sem que haja pagamento, mediante decreto ou lei. Nesse caso, o Estado de origem decide perdo


Gemma3 da Google usada para desenvolvimento.

In [ ]:
# Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
# Criada previamente no hugging face.
hf_token = userdata.get('huggingface_hub')

# Autenticar no Hugging Face para acessar modelos lá disponíveis.
huggingface_hub.login(token=hf_token)

# Inicializar o cliente da API (opcional, se não for usar a API do Google GenAI)
client_ai = genai.Client(api_key=google_api_key)

# Modelo para rodar com GPUs:T4. Limite de requisições gratuitas -> 250.
model_id = 'google/gemma-4-31B-it'
#model_id = "google/gemma-3-4b-it" # Exemplo com a versão 4B Instruct

# O modelo selecionado tem 27 bilhões de parâmetros e não é possível executar na GPU T4 do Google Colab,
# por isso o uso da tokenização.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Configuração da quantização para caber na GPU T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Definição do Modelo 27B
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)

# Criar uma lista vazia, para guardar as respostas, por questão de performance.
gemini_responses = []

# Repetição para percorrer todo Dataframe.
for index, row in df_question_and_guidelines.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    papel = row['system']
    categoria = row['category']
    contexto = row['statement']
    instrucao = row['turns']

    # Preparação do prompt em Markdown
    prompt_usuario = f"""
    ### PAPEL:
    {papel}

    ### ÁREA DE ATUAÇÃO:
    {categoria}

    ### CONTEXTO:
    {contexto}

    ### INSTRUÇÃO:
    {instrucao}
    """

    # Preparação da template para o chat.
    messages = [{"role": "user", "content": prompt_usuario}]
    prompt_formatado = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Geração da resposta.
    inputs = tokenizer(prompt_formatado, return_tensors="pt").to("cuda")

    with torch.no_grad(): # Economiza memória ao não calcular gradientes
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1, # Mantém a precisão técnica
            do_sample=False  # Garante respostas mais determinísticas para questões técnicas.
        )


    # Decodificação e Limpeza
    full_respense = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extrai apenas a parte que vem depois da tag do modelo
    clear_response = full_respense.split("model\n")[-1].strip()

    # Armazenar o resultado em uma lista.
    gemini_responses.append({"question": questao, "response": clear_response})
    print(f"Questão {questao} processada com sucesso.")

# Fechar conexão com a IA (somente se você a usou ativamente)
client_ai.close()

# Para melhor visualização, converter as respostas em um Dataframe.
df_gemini_response = pd.DataFrame(gemini_responses)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-3-27b-it.
401 Client Error. (Request ID: Root=1-69ce6943-34517def520654a1368c34de;f24cf647-a9ee-40ab-b4c6-0dd4177ace26)

Cannot access gated repo for url https://huggingface.co/google/gemma-3-27b-it/resolve/main/config.json.
Access to model google/gemma-3-27b-it is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
# Exibir as 5 primeiras linhas de respostas.
df_gemini_response.head()

,question,response
0,31,sdk_http_response=HttpResponse(\n headers=<di...
1,32,sdk_http_response=HttpResponse(\n headers=<di...
2,33,sdk_http_response=HttpResponse(\n headers=<di...
3,34,sdk_http_response=HttpResponse(\n headers=<di...
4,35,sdk_http_response=HttpResponse(\n headers=<di...


In [ ]:
# Objetos mais relevantes até aqui.

# DataFrame com perguntas e respostas da linha guia.
df_questions_and_guidelines.info()

# Dataframe com as respostas do gemini, com o campo question para relacionar com o dataframe das perguntas.
df_gemini_response.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   question     10 non-null     int64  
 1   question_id  10 non-null     object 
 2   category     10 non-null     object 
 3   statement    10 non-null     object 
 4   turns        10 non-null     object 
 5   values       10 non-null     object 
 6   system       10 non-null     object 
 7   answer_id    10 non-null     object 
 8   model_id     10 non-null     object 
 9   choices      10 non-null     object 
 10  tstamp       2 non-null      float64
dtypes: float64(1), int64(1), object(9)
memory usage: 1012.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  10 non-null     int64 
 1   response  10 non-null     object
dtypes: int64(1), object(1)
mem

In [ ]:
import huggingface_hub
from google.colab import userdata
from transformers import pipeline

# Get the Hugging Face token from Colab Secrets
hugging_colab_token = userdata.get('hugging_colab')

# Authenticate with Hugging Face
try:
    huggingface_hub.login(token=hugging_colab_token)
    print("Successfully logged into Hugging Face.")
except Exception as e:
    print(f"Error logging into Hugging Face: {e}")

# Define the model to be downloaded/loaded
model_id = "dominguesm/legal-bert-base-cased-ptbr"

# Load the model using a pipeline
try:
    print(f"Attempting to load model: {model_id}")
    legal_bert_pipeline = pipeline("text-classification", model=model_id, tokenizer=model_id)
    print(f"Model '{model_id}' loaded successfully into 'legal_bert_pipeline'.")
except Exception as e:
    print(f"Error loading model '{model_id}': {e}")
    legal_bert_pipeline = None


Successfully logged into Hugging Face.
Attempting to load model: dominguesm/legal-bert-base-cased-ptbr


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/834 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/513 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Model 'dominguesm/legal-bert-base-cased-ptbr' loaded successfully into 'legal_bert_pipeline'.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Install transformers library
!pip install -q transformers

from transformers import pipeline

# Load the jurisBERT model
try:
    # Using the adalpa/jurisbert model as it's a common one for legal text
    jurisbert_pipeline = pipeline("text-classification", model="google-bert/bert-base-uncased", tokenizer="google-bert/bert-base-uncased")
except Exception as e:
    print(f"Error loading jurisBERT model: {e}")
    print("Please ensure the model name 'google-bert/bert-base-uncased' is correct or try another legal BERT model if this fails.")
    jurisbert_pipeline = None

jurisbert_responses = []

if jurisbert_pipeline:
    print("Processing questions with jurisBERT...")
    # Iterate through df_my_questions and get predictions
    for index, row in df_my_questions.iterrows():
        question_id = row['question'] # Assuming 'question' column identifies the question
        statement = row['statement']
        try:
            # Process the statement with the jurisBERT pipeline
            # The output structure may vary depending on the specific model's task
            result = jurisbert_pipeline(statement)
            jurisbert_responses.append({"question": question_id, "response": result})
            print(f"Questão {question_id} processada com sucesso por jurisBERT.")
        except Exception as e:
            print(f"Error processing question {question_id} with jurisBERT: {e}")
            jurisbert_responses.append({"question": question_id, "response": f"Error: {e}"})
else:
    print("jurisBERT pipeline could not be initialized. Skipping processing.")

# Convert the list of responses to a DataFrame for consistency, if needed
jurisbert_df_responses = pd.DataFrame(jurisbert_responses)
print("Processing complete. jurisbert_df_responses contains the results.")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (861 > 512). Running this sequence through the model will result in indexing errors


Processing questions with jurisBERT...
Error processing question 31 with jurisBERT: The size of tensor a (861) must match the size of tensor b (512) at non-singleton dimension 1
Questão 32 processada com sucesso por jurisBERT.
Questão 33 processada com sucesso por jurisBERT.
Questão 34 processada com sucesso por jurisBERT.
Questão 35 processada com sucesso por jurisBERT.
Error processing question 36 with jurisBERT: The size of tensor a (911) must match the size of tensor b (512) at non-singleton dimension 1
Questão 37 processada com sucesso por jurisBERT.
Questão 38 processada com sucesso por jurisBERT.
Questão 39 processada com sucesso por jurisBERT.
Questão 40 processada com sucesso por jurisBERT.
Processing complete. jurisbert_df_responses contains the results.


In [ ]:
legal_bert_results = []

if legal_bert_pipeline:
    print("Processing questions with legal_bert_pipeline...")
    for index, row in df_my_questions.iterrows():
        question_id = row['question'] # Assuming 'question' column identifies the question
        statement = row['statement']
        try:
            # Process the statement with the legal_bert_pipeline
            # The output structure may vary depending on the specific model's task
            result = legal_bert_pipeline(statement)
            legal_bert_results.append({"question": question_id, "classification_result": result})
            print(f"Questão {question_id} processada com sucesso por legal_bert_pipeline.")
        except Exception as e:
            print(f"Error processing question {question_id} with legal_bert_pipeline: {e}")
            legal_bert_results.append({"question": question_id, "classification_result": f"Error: {e}"})
else:
    print("legal_bert_pipeline could not be initialized. Skipping processing.")

df_legal_bert_results = pd.DataFrame(legal_bert_results)
print("Processing complete. df_legal_bert_results contains the classification results.")


,question,classification_result
0,31,Error: The size of tensor a (520) must match t...
1,32,"[{'label': 'LABEL_1', 'score': 0.5824732184410..."
2,33,"[{'label': 'LABEL_1', 'score': 0.5932129621505..."
3,34,"[{'label': 'LABEL_1', 'score': 0.6145867109298..."
4,35,"[{'label': 'LABEL_1', 'score': 0.6011602878570..."
5,36,"[{'label': 'LABEL_1', 'score': 0.5858585834503..."
6,37,"[{'label': 'LABEL_1', 'score': 0.5701338648796..."
7,38,"[{'label': 'LABEL_1', 'score': 0.6021652817726..."
8,39,"[{'label': 'LABEL_1', 'score': 0.5616039037704..."
9,40,"[{'label': 'LABEL_1', 'score': 0.5544844865798..."


In [ ]:
display(df_legal_bert_results.head())

In [ ]:
def extract_legal_bert_response(result):
    if isinstance(result, list) and result and 'label' in result[0]:
        return result[0]['label']
    elif isinstance(result, str) and result.startswith('Error:'):
        return result
    return None

df_legal_bert_results['legal_bert_response'] = df_legal_bert_results['classification_result'].apply(extract_legal_bert_response)

print("DataFrame with new 'legal_bert_response' column:")
display(df_legal_bert_results.head())

DataFrame with new 'legal_bert_response' column:


,question,classification_result,legal_bert_response
0,31,Error: The size of tensor a (520) must match t...,Error: The size of tensor a (520) must match t...
1,32,"[{'label': 'LABEL_1', 'score': 0.5824732184410...",LABEL_1
2,33,"[{'label': 'LABEL_1', 'score': 0.5932129621505...",LABEL_1
3,34,"[{'label': 'LABEL_1', 'score': 0.6145867109298...",LABEL_1
4,35,"[{'label': 'LABEL_1', 'score': 0.6011602878570...",LABEL_1


In [ ]:
# Iniciar o cliente da IA
def client_ai_instance(groq_api_key):
  # Inicializar o cliente da API.
  client_ai = OpenAI(
      base_url="https://api.groq.com/openai/v1",
      api_key=groq_api_key
  )
  return client_ai

def markdown_prepare(papel, categoria, contexto, instrucao):
  # Preparação do prompt em Markdown
  prompt_usuario = f"""
  ### Papel:
  {papel}

  ### Área de atuação:
  {categoria}

  ### Contexto:
  {contexto}

  ### Instrução:
  {instrucao}
  """
  return prompt_usuario

def questions_submit(client_ai, model_id, df_questions):
    # Criar lista vazia para guardar as respostas.
    responses = []

    # Repetição para percorrer todo Dataframe.
    for index, row in df_questions.iterrows():
      # preenchimento dos parâmetros da pergunta, com base na linha corrente.
      questao = row['question']
      papel = row['system']
      categoria = row['category']
      contexto = row['statement']
      instrucao = row['turns']

      # Preencher prompt do usuário.
      prompt_usuario = markdown_prepare(papel, categoria, contexto, instrucao)

      # Realizar consulta a IA.
      completion = client_ai.chat.completions.create(
          model= model_id,
          messages=[
            # Por questões de sintaxes informo a role, pois é um campo obrigatório,
            # porém o conteúdo é somente o do Markdown.
            {"role": "user", "content": prompt_usuario}
          ],
          temperature=0.1 # Conservador.
      )
      response = completion.choices[0].message.content

      # Armazenar o resultado em uma lista.
      responses.append({"question": questao, "response": response})
      print(f"Questão {questao} processada com sucesso.")

    # Fechar conexão com a IA (somente se você a usou ativamente)
    client_ai.close()

    # por motivo de performance as repostas foram adicionadas a uma lista.
    # No retorno a lista é convertida para um dataframe.
    return pd.DataFrame(responses)